# Student Performance Analysis

## Project Overview
Analyze student academic data to understand how attendance, study time, family background, and school variables relate to performance outcomes.

## Learning Objectives
- Explore relationships between study habits and academic performance
- Analyze the impact of family and socioeconomic factors on grades
- Identify high-risk and high-performing student profiles
- Quantify relative importance of different performance drivers

## Problem Statement
School administrators and educators need data-driven insight into what factors most strongly predict student success, so they can design targeted interventions for struggling students.

## Why This Project Matters
Understanding academic performance drivers helps allocate tutoring resources, design effective study programs, and identify at-risk students before they fall behind.

## Dataset Overview
Synthetic dataset of ~1,500 students with demographics, study habits, family background, school features, and final exam scores.

## Dataset Source and License Notes
- Synthetic data inspired by UCI Student Performance datasets
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 1500
gender = np.random.choice(['Male', 'Female'], n, p=[0.48, 0.52])
age = np.random.choice(range(15, 20), n, p=[0.15, 0.30, 0.28, 0.17, 0.10])
study_hours = np.clip(np.random.exponential(2.5, n), 0.5, 12).round(1)
attendance_pct = np.clip(np.random.normal(82, 12, n), 30, 100).round(1)
parent_edu = np.random.choice(['None', 'Primary', 'Secondary', 'Higher'], n, p=[0.05, 0.20, 0.45, 0.30])
family_income = np.random.choice(['Low', 'Medium', 'High'], n, p=[0.30, 0.45, 0.25])
internet_access = np.random.choice(['Yes', 'No'], n, p=[0.72, 0.28])
extra_activities = np.random.choice(['Yes', 'No'], n, p=[0.40, 0.60])
school_type = np.random.choice(['Public', 'Private'], n, p=[0.65, 0.35])
tutoring = np.random.choice(['Yes', 'No'], n, p=[0.25, 0.75])

# Score driven by study hours, attendance, parent edu, income
edu_bonus = {'None': -8, 'Primary': -3, 'Secondary': 2, 'Higher': 7}
income_bonus = {'Low': -5, 'Medium': 0, 'High': 5}
base_score = (40
              + study_hours * 3.5
              + attendance_pct * 0.2
              + np.array([edu_bonus[e] for e in parent_edu])
              + np.array([income_bonus[i] for i in family_income])
              + np.where(internet_access == 'Yes', 2, -2)
              + np.where(tutoring == 'Yes', 4, 0)
              + np.random.normal(0, 6, n))
final_score = np.clip(base_score, 0, 100).round(1)
grade = pd.cut(final_score, bins=[0, 40, 55, 70, 85, 100], labels=['F', 'D', 'C', 'B', 'A'])

df = pd.DataFrame({
    'StudentID': [f'S{i:04d}' for i in range(n)],
    'Gender': gender, 'Age': age, 'StudyHoursPerWeek': study_hours,
    'AttendancePct': attendance_pct, 'ParentEducation': parent_edu,
    'FamilyIncome': family_income, 'InternetAccess': internet_access,
    'ExtraActivities': extra_activities, 'SchoolType': school_type,
    'Tutoring': tutoring, 'FinalScore': final_score, 'Grade': grade
})
print(f'Dataset: {df.shape}')
print(f'Score range: {df["FinalScore"].min()} — {df["FinalScore"].max()}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nGrade distribution:\n{df["Grade"].value_counts().sort_index()}')

## Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['FinalScore'].hist(bins=30, ax=axes[0], edgecolor='black', color='steelblue')
axes[0].set_title('Final Score Distribution')
axes[0].axvline(df['FinalScore'].mean(), color='red', ls='--', label=f'Mean={df["FinalScore"].mean():.1f}')
axes[0].legend()
df['Grade'].value_counts().sort_index().plot.bar(ax=axes[1], edgecolor='black', color='teal')
axes[1].set_title('Grade Distribution')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'score_distribution.png'), dpi=100, bbox_inches='tight')
plt.show()

## Study Hours and Attendance Impact

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(df['StudyHoursPerWeek'], df['FinalScore'], alpha=0.3, s=15)
z = np.polyfit(df['StudyHoursPerWeek'], df['FinalScore'], 1)
axes[0].plot(np.sort(df['StudyHoursPerWeek']), np.polyval(z, np.sort(df['StudyHoursPerWeek'])), 'r-', lw=2)
axes[0].set_xlabel('Study Hours / Week')
axes[0].set_ylabel('Final Score')
axes[0].set_title('Study Hours vs Score')

axes[1].scatter(df['AttendancePct'], df['FinalScore'], alpha=0.3, s=15)
z2 = np.polyfit(df['AttendancePct'], df['FinalScore'], 1)
axes[1].plot(np.sort(df['AttendancePct']), np.polyval(z2, np.sort(df['AttendancePct'])), 'r-', lw=2)
axes[1].set_xlabel('Attendance %')
axes[1].set_ylabel('Final Score')
axes[1].set_title('Attendance vs Score')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'study_attendance.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'Correlation — Study Hours: {df["StudyHoursPerWeek"].corr(df["FinalScore"]):.3f}')
print(f'Correlation — Attendance: {df["AttendancePct"].corr(df["FinalScore"]):.3f}')

## Family Background Effects

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
edu_order = ['None', 'Primary', 'Secondary', 'Higher']
sns.boxplot(data=df, x='ParentEducation', y='FinalScore', order=edu_order, ax=axes[0])
axes[0].set_title('Score by Parent Education')
inc_order = ['Low', 'Medium', 'High']
sns.boxplot(data=df, x='FamilyIncome', y='FinalScore', order=inc_order, ax=axes[1])
axes[1].set_title('Score by Family Income')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'family_background.png'), dpi=100, bbox_inches='tight')
plt.show()
print('Mean score by parent education:')
print(df.groupby('ParentEducation')['FinalScore'].mean().reindex(edu_order).round(1))

## School and Resource Effects

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
sns.boxplot(data=df, x='SchoolType', y='FinalScore', ax=axes[0])
axes[0].set_title('Score by School Type')
sns.boxplot(data=df, x='InternetAccess', y='FinalScore', ax=axes[1])
axes[1].set_title('Score by Internet Access')
sns.boxplot(data=df, x='Tutoring', y='FinalScore', ax=axes[2])
axes[2].set_title('Score by Tutoring')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'school_resources.png'), dpi=100, bbox_inches='tight')
plt.show()

## Gender and Age Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='Gender', y='FinalScore', ax=axes[0])
axes[0].set_title('Score by Gender')
sns.boxplot(data=df, x='Age', y='FinalScore', ax=axes[1])
axes[1].set_title('Score by Age')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'gender_age.png'), dpi=100, bbox_inches='tight')
plt.show()

## Correlation Heatmap

In [ ]:
numeric_df = df[['StudyHoursPerWeek', 'AttendancePct', 'Age', 'FinalScore']].copy()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Numeric Feature Correlations')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'correlation_heatmap.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **Study hours** is the strongest predictor of final score — each extra hour adds ~3-4 points
- **Attendance** matters but less strongly — consistent attendance supports learning
- **Parent education** creates a ~15 point gap between 'None' and 'Higher' — socioeconomic equity needs attention
- **Tutoring** adds ~4 points on average — cost-effective for borderline students
- **Internet access** shows a modest positive effect — digital divide is real
- **Gender** differences are minimal — both groups perform similarly

## Limitations
- Synthetic data with linear relationships
- No longitudinal tracking
- No teacher quality or curriculum data
- No motivation or mental health variables
- Small number of school-level variables

## How to Improve This Project
- Add multi-semester tracking for trend analysis
- Include teacher evaluations and class size
- Add standardized test scores as validation
- Include motivation surveys and learning style data
- Build predictive models for early intervention

## Production Considerations
- Early warning dashboards for at-risk students
- Automated alerts for declining attendance or grades
- Resource allocation models for tutoring programs
- Parent engagement tools based on risk profiles

## Common Mistakes
- Assuming correlation equals causation (tutoring vs self-selection)
- Ignoring socioeconomic confounders
- Treating grades as interval when they are ordinal
- Not segmenting analysis by relevant subgroups

## Mini Challenge / Exercises
1. What is the average score difference between students with tutoring vs without, controlling for study hours (above/below median)?
2. Which combination of parent education + family income produces the highest average score?
3. Calculate the pass rate (score >= 40) by attendance quartile.
4. If attendance increased by 10% for all students below 70% attendance, estimate the score impact.

## Final Summary / Key Takeaways
- Study hours and attendance are actionable levers for improving performance
- Family background creates significant baseline advantages — equity programs matter
- Tutoring is effective and should be targeted at borderline students
- Data-driven student support systems can identify at-risk students early
- Simple analytics can guide meaningful educational interventions